# Model Training

Train ML models for price prediction.

In [ ]:
# Restart kernel to reload modules
import importlib
import sys

# Remove cached modules
for module in list(sys.modules.keys()):
    if 'src' in module:
        del sys.modules[module]

import pandas as pd
sys.path.append('..')

from src.utils.config import DEFAULT_TICKERS, RAW_DATA_DIR
from src.modeling.train_model import (
    prepare_features, 
    train_stacked_model, 
    train_improved_stacked_model,
    train_tuned_stacked_model,
    tune_xgboost_hyperparameters,
    save_model
)
from src.modeling.evaluate_model import evaluate_classification, plot_confusion_matrix, plot_feature_importance

print("✓ Imports loaded successfully")

ImportError: cannot import name 'train_improved_stacked_model' from 'src.modeling.train_model' (c:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\notebooks\..\src\modeling\train_model.py)

In [15]:
# Check all available intervals
print("Available raw data intervals:")
print("-" * 50)
for ticker in DEFAULT_TICKERS:
    print(f"\n{ticker}:")
    for interval in ["1d", "5m", "1m"]:
        suffix = f"_{interval}" if interval != "1d" else ""
        filepath = RAW_DATA_DIR / f"{ticker}{suffix}.csv"
        if filepath.exists():
            print(f"  ✓ {interval}")
        else:
            print(f"  ✗ {interval}: not found")

Available raw data intervals:
--------------------------------------------------

RELIANCE.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

TCS.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

INFY.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

HDFCBANK.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m

ICICIBANK.NS:
  ✓ 1d
  ✓ 5m
  ✓ 1m


## Check Available Data Intervals

In [16]:
%pip install seaborn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# Load features (support interval-specific feature files)
from pathlib import Path
# Use project tickers (DEFAULT_TICKERS) instead of AAPL
ticker = DEFAULT_TICKERS[1]  # change index to pick another project ticker
suffixes = ["", "_5m", "_1m"]

candidates = []
for s in suffixes:
    candidates.extend([
        Path('../data/indicators') / f'{ticker}{s}_features.csv',
        Path('../../data/indicators') / f'{ticker}{s}_features.csv',
        Path('../../../data/indicators') / f'{ticker}{s}_features.csv',
        Path('../../../../data/indicators') / f'{ticker}{s}_features.csv',
        Path('../../../../data') / 'indicators' / f'{ticker}{s}_features.csv',
    ])

for p in candidates:
    if p.exists():
        data = pd.read_csv(p, index_col=0, parse_dates=True)
        print(f'Loaded features from: {p}')
        break
else:
    raise FileNotFoundError(
        f'Could not find {ticker}_features.csv. Checked paths: {', '.join(str(x) for x in candidates)}'
    )

# Prepare data
X_train, X_test, y_train, y_test, scaler, features = prepare_features(data)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Features: {len(features)}' )

Loaded features from: ..\data\indicators\TCS.NS_features.csv
2025-12-19 12:10:14 - src.modeling.train_model - INFO - Prepared data | Train: (862, 39) | Test: (216, 39)
Training samples: 862
Test samples: 216
Features: 39


In [18]:
# Train stacked model (RandomForest + XGBoost)
model = train_stacked_model(X_train, y_train, rf_estimators=100, xgb_estimators=100)

2025-12-19 12:10:18 - src.modeling.train_model - INFO - Training stacked model (RandomForest + XGBoost)...
2025-12-19 12:10:24 - src.modeling.train_model - INFO - Stacked model training completed


In [19]:
# Evaluate
y_pred = model.predict(X_test)
metrics = evaluate_classification(y_test, y_pred, model_name='Stacked (RF + XGBoost)')

2025-12-19 12:10:26 - src.modeling.evaluate_model - INFO - 
Stacked (RF + XGBoost) Performance:
2025-12-19 12:10:26 - src.modeling.evaluate_model - INFO - Accuracy: 0.4954
2025-12-19 12:10:26 - src.modeling.evaluate_model - INFO - Precision: 0.5024
2025-12-19 12:10:26 - src.modeling.evaluate_model - INFO - Recall: 0.9455
2025-12-19 12:10:26 - src.modeling.evaluate_model - INFO - F1 Score: 0.6562
2025-12-19 12:10:26 - src.modeling.evaluate_model - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.03      0.05       106
           1       0.50      0.95      0.66       110

    accuracy                           0.50       216
   macro avg       0.42      0.49      0.35       216
weighted avg       0.42      0.50      0.36       216



In [20]:
# Save model
save_model(model, scaler, features, ticker)
print('Model saved successfully!')

2025-12-19 12:10:33 - src.modeling.train_model - INFO - Saved model artifacts for TCS.NS
Model saved successfully!


In [21]:
# Verify model composition
print("Stacked Model Components:")
print(f"Base Learners: {model.estimators_}")
print(f"Final Estimator: {model.final_estimator_}")

Stacked Model Components:
Base Learners: [RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42), XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=1,
              num_parallel_tree=None, ...)]
Final Estimator: LogisticRegression(max_iter=1000)


## Improve Model Accuracy

Try these optimization strategies to increase accuracy:

In [ ]:
# Option 1: Train improved stacked model with enhanced hyperparameters
from sklearn.metrics import accuracy_score

model_improved = train_improved_stacked_model(X_train, y_train)
y_pred_improved = model_improved.predict(X_test)

# Compare accuracies
acc_original = accuracy_score(y_test, y_pred)
acc_improved = accuracy_score(y_test, y_pred_improved)

print(f"Original Stacked Model Accuracy: {acc_original:.4f}")
print(f"Improved Stacked Model Accuracy: {acc_improved:.4f}")
print(f"Improvement: {(acc_improved - acc_original)*100:.2f}%")

# Evaluate improved model
metrics_improved = evaluate_classification(y_test, y_pred_improved, model_name='Improved Stacked (RF + XGB + GB)')

NameError: name 'StackingClassifier' is not defined

In [ ]:
# Option 2: Use GridSearchCV to find best XGBoost hyperparameters
best_xgb, best_cv_score = tune_xgboost_hyperparameters(X_train, y_train)

# Then train stacked model with tuned hyperparameters
model_tuned, tuned_accuracy = train_tuned_stacked_model(X_train, y_train, X_test, y_test)
y_pred_tuned = model_tuned.predict(X_test)

print(f"\n--- Accuracy Comparison ---")
print(f"Original Model:  {acc_original:.4f}")
print(f"Improved Model:  {acc_improved:.4f}")
print(f"Tuned Model:     {tuned_accuracy:.4f}")

# Evaluate tuned model
metrics_tuned = evaluate_classification(y_test, y_pred_tuned, model_name='Tuned Stacked (RF + XGB + GB)')

### Other Strategies to Improve Accuracy:

1. **Feature Engineering**: Create new features or remove irrelevant ones
2. **Data Preprocessing**: Better handling of outliers, normalization
3. **Class Imbalance**: Use SMOTE or adjust class weights further
4. **Ensemble**: Add more diverse base learners (SVM, Neural Networks)
5. **Target Engineering**: Ensure target variable is well-defined
6. **Feature Scaling**: Try different scaling methods